# Sparse linear regression — results inspection

Loads `.pt` files written by `sazz.scripts.linear_regression --save` for
a sparse high-D run, mirrors the workflow of `uci_results.ipynb`, and
shows the sparse-specific diagnostics:

1. Metrics table (β-RMSE, predictive RMSE, F1 at 95% CI, σ-ratio, P(=0))
2. Two-panel coefficient plot (signals + nulls)
3. Inclusion-probability ROC for the Sticky variants
4. Variable selection comparison at multiple decision rules
5. Correctness check vs the analytic Gaussian-prior posterior

To produce a run on disk:

```
python -m sazz.scripts.linear_regression \
    --N 2000 --D 200 --n-signals 10 --signal-scale 2.0 \
    --intercept 0 --noise-std 1.0 --prior-std 5.0 --lik-noise-std 1.0 \
    --save --name sparse_main
```

Then change `RUN_NAME` below to point at it.

## 0. Setup

In [ ]:
import os, sys, json
from pathlib import Path

if Path.cwd().name == "notebooks":
    os.chdir("..")

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

from sklearn.metrics import f1_score, precision_score, recall_score, roc_curve, auc

torch.set_default_dtype(torch.float64)

# ---- Pick the run ----
RESULTS_DIR = Path("results/linear_regression")
RUN_NAME    = "N2000_D200_K8_Gauss_pli_seed0"     # subdir under RESULTS_DIR

run_dir = RESULTS_DIR / RUN_NAME
print(f"Looking in {run_dir}")
print(f"  found: {sorted(p.name for p in run_dir.glob('*.pt'))}")

## 1. Load saved runs

Each `.pt` is `{name, samples, wall, config}`. The `metrics.json`
sidecar additionally holds `true_coefs`, `is_signal`, and the metric
rows recorded at run time.

In [ ]:
def load_runs(run_dir: Path) -> tuple[list[dict], dict]:
    """Load all sampler .pt files plus the metrics.json sidecar."""
    runs = []
    for pt_path in sorted(run_dir.glob("*.pt")):
        payload = torch.load(pt_path, weights_only=False)
        runs.append({
            "name":    payload["name"],
            "samples": payload["samples"].cpu().numpy(),
            "wall":    payload.get("wall"),
            "config":  payload["config"],
        })
    with open(run_dir / "metrics.json") as f:
        meta = json.load(f)
    return runs, meta


runs, meta = load_runs(run_dir)
true_coefs = np.array(meta["true_coefs"])
is_signal  = np.array(meta["is_signal"])
cfg        = meta["config"]

# Restore display style per sampler — matches the script's choices
STYLE = {
    "Analytic":       ("k",  "D"),
    "NUTS-Gaussian":  ("C2", "D"),
    "NUTS-horseshoe": ("C4", "P"),
    "Boom":           ("C0", "o"),
    "Sticky-Boom":    ("C1", "s"),
    "ZZ":             ("C9", "v"),
    "Sticky-ZZ":      ("C3", "^"),
}
for r in runs:
    r["color"], r["marker"] = STYLE.get(r["name"], ("C5", "o"))
    r["sticky"] = "sticky" in r["name"].lower() or r["name"].startswith("Sticky")

print(f"Loaded {len(runs)} samplers: {[r['name'] for r in runs]}")
print(f"D={cfg['D']}, K={cfg['n_signals']}, N={cfg['N']}, "
      f"signals_actual={int(is_signal.sum())}/{len(true_coefs)}")

## 2. Recompute metrics from saved samples

We recompute everything from the persisted samples rather than trusting
the metric rows from the original run. Same fresh-DGP test set as the
script — using `seed + 1` to keep them aligned.

In [ ]:
def support_via_ci(samples: np.ndarray, alpha: float = 0.05) -> np.ndarray:
    lo = np.quantile(samples, alpha / 2,     axis=0)
    hi = np.quantile(samples, 1 - alpha / 2, axis=0)
    return (lo > 0) | (hi < 0)


def support_via_pzero(samples: np.ndarray, threshold: float = 0.5) -> np.ndarray:
    p_zero = (np.abs(samples) < 1e-8).mean(0)
    return p_zero <= threshold


# Rebuild test set deterministically from the saved config
N, D = cfg["N"], cfg["D"] + 1

test_rng = np.random.default_rng(cfg["seed"] + 1)
X_te = test_rng.normal(size=(N, D))
X_te = (X_te - X_te.mean(0)) / X_te.std(0)
# beta_te = true_coefs[1:] if has_int else true_coefs
# int_te  = true_coefs[0] if has_int else 0.0
y_te_clean = X_te @ true_coefs



def metrics_for(r: dict, sd_ref: np.ndarray | None) -> dict:
    s = r["samples"]
    mu, sd = s.mean(0), s.std(0)
    pred_mean = (s @ X_te.T).mean(0)
    p_zero = (np.abs(s) < 1e-8).mean(0) if r["sticky"] else None
    return {
        "wall":        r.get("wall"),
        "n_draws":     s.shape[0],
        "beta_rmse":   float(np.sqrt(((mu - true_coefs) ** 2).mean())),
        "pred_rmse":   float(np.sqrt(((pred_mean - y_te_clean) ** 2).mean())),
        "f1_ci":       float(f1_score(is_signal, support_via_ci(s), zero_division=0)),
        "sigma_ratio": (float((sd[is_signal] / sd_ref[is_signal]).mean())
                        if sd_ref is not None and is_signal.any() else float("nan")),
        "p0_nulls":    (float(p_zero[~is_signal].mean())
                        if (p_zero is not None and (~is_signal).any()) else float("nan")),
    }


# Reference σ for σ-ratio: prefer Analytic, else NUTS-Gaussian
sd_ref = next((r["samples"].std(0) for r in runs
               if r["name"] in ("NUTS-Gaussian")), None)

df = pd.DataFrame(
    [{"sampler": r["name"], **metrics_for(r, sd_ref)} for r in runs]
).set_index("sampler")
df.round(4)

## 3. Coefficient plots

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# ===========================================================================
# Plot 1 — Signal coefficients as a small-multiples grid
# ===========================================================================

sig = np.where(is_signal)[0]
nul = np.where(~is_signal)[0]
n_sig = len(sig)

# Auto-pick a near-square grid (e.g. 11 sigs -> 3x4, 9 -> 3x3, 6 -> 2x3)
ncols = int(np.ceil(np.sqrt(n_sig)))
nrows = int(np.ceil(n_sig / ncols))

fig, axes = plt.subplots(nrows, ncols, figsize=(3.2 * ncols, 2.6 * nrows),
                         squeeze=False)
axes_flat = axes.flatten()

method_names = [r["name"] for r in runs]
x_pos = np.arange(len(runs))

for panel_i, coord in enumerate(sig):
    ax = axes_flat[panel_i]
    truth = true_coefs[coord]

    for j, r in enumerate(runs):
        samples_c = r["samples"][:, coord]
        parts = ax.violinplot(samples_c, positions=[j], widths=0.75,
                              showextrema=False, showmedians=False)
        for body in parts["bodies"]:
            body.set_facecolor(r["color"])
            body.set_edgecolor(r["color"])
            body.set_alpha(0.6)
        # Median tick
        ax.scatter([j], [np.median(samples_c)], color=r["color"],
                   marker=r["marker"], s=20, edgecolor="white",
                   linewidths=0.5, zorder=4)

    # Truth line — horizontal so it spans all methods
    ax.axhline(truth, color="red", lw=1.5, ls="--", alpha=0.85,
               label="true" if panel_i == 0 else None)
    ax.axhline(0, color="grey", lw=0.5, alpha=0.5)

    ax.set_xticks(x_pos)
    ax.set_xticklabels(method_names, rotation=45, ha="right", fontsize=7)
    ax.set_title(rf"$\beta_{{{coord}}}$  (true = {truth:.2f})", fontsize=10)
    ax.tick_params(axis="y", labelsize=8)
    ax.grid(axis="y", alpha=0.25)

# Hide any leftover empty panels
for k in range(n_sig, len(axes_flat)):
    axes_flat[k].set_visible(False)

# Single shared legend at the figure level
handles = [plt.Line2D([], [], marker=r["marker"], color=r["color"],
                      linestyle="", markersize=7, label=r["name"])
           for r in runs]
handles.append(plt.Line2D([], [], color="red", lw=1.5, ls="--", label="true"))
fig.legend(handles=handles, loc="lower center",
           ncol=min(len(handles), 7), frameon=False,
           bbox_to_anchor=(0.5, -0.02))

fig.suptitle(f"Signal coefficients — D={D}, K={cfg['n_signals']}, "
             f"thinning={cfg['thinning']}", fontsize=12, y=1.0)
plt.tight_layout(rect=[0, 0.03, 1, 0.98])
plt.show()


# ===========================================================================
# Plot 2 — Null shrinkage strip per method  (unchanged from before)
# ===========================================================================

fig, ax = plt.subplots(figsize=(11, 0.55 * len(runs) + 1.2))
ordered = sorted(runs, key=lambda r: (r["name"] in ("NUTS-Gaussian", "NUTS-horseshoe"),
                                      r["name"]))
rng = np.random.default_rng(0)
for i, r in enumerate(ordered):
    means_on_nulls = r["samples"].mean(0)[nul]
    jitter = rng.uniform(-0.20, 0.20, size=len(nul))
    ax.scatter(means_on_nulls, np.full_like(means_on_nulls, i) + jitter,
               color=r["color"], marker=r["marker"],
               s=18, alpha=0.55, edgecolor="none")
    q25, q50, q75 = np.quantile(means_on_nulls, [0.25, 0.50, 0.75])
    ax.plot([q25, q75], [i, i], color="black", lw=2.5, solid_capstyle="butt", zorder=4)
    ax.scatter([q50], [i], color="black", marker="|", s=140, lw=2.5, zorder=5)

ax.axvline(0, color="red", lw=1, alpha=0.8)
ax.set_yticks(np.arange(len(ordered)))
ax.set_yticklabels([r["name"] for r in ordered])
ax.set_xlabel("posterior mean on null coordinates")
ax.set_title(f"Null shrinkage across {len(nul)} true zeros  "
             f"(black bar = IQR, tick = median)")
ax.invert_yaxis()
ax.grid(axis="x", alpha=0.3)
plt.tight_layout(); plt.show()

## 4. Inclusion-probability ROC

For the Sticky variants, treat `1 − P(=0)` as a score for "this
coordinate is a true signal" and trace the ROC curve. AUC near 1.0
means the inclusion probability separates signals from nulls cleanly.

In [ ]:
sticky_runs = [r for r in runs if r["sticky"]]
if not sticky_runs:
    print("No sticky samplers in this run.")
else:
    fig, ax = plt.subplots(figsize=(6, 5))
    for r in sticky_runs:
        s = r["samples"]
        is_sig = is_signal
        p_zero = (np.abs(s) < 1e-8).mean(0)
        incl = 1.0 - p_zero
        fpr, tpr, _ = roc_curve(is_sig.astype(int), incl)
        ax.plot(fpr, tpr, color=r["color"], lw=1.5,
                label=f"{r['name']}  (AUC = {auc(fpr, tpr):.3f})")
    ax.plot([0, 1], [0, 1], color="grey", lw=0.5, ls="--", label="chance")
    ax.set_xlabel("false positive rate")
    ax.set_ylabel("true positive rate")
    ax.set_title("Inclusion-probability ROC — Sticky variants")
    ax.legend(loc="lower right", frameon=False)
    plt.tight_layout(); plt.show()

## 5. Variable selection at multiple rules

- **95% CI excludes 0**: the only fair rule across all methods (Sticky,
  horseshoe, Gaussian-prior all support it).
- **P(=0) ≤ 0.5**: Sticky-only — exact-zero mode rule.

Reports TP / FP / FN and the standard precision / recall / F1 trio.

In [ ]:
def report(name: str, sel: np.ndarray, rule: str, is_sig: np.ndarray):
    tp = int((sel & is_sig).sum())
    fp = int((sel & ~is_sig).sum())
    fn = int((~sel & is_sig).sum())
    p  = precision_score(is_sig, sel, zero_division=0)
    r  = recall_score(is_sig, sel, zero_division=0)
    f1 = f1_score(is_sig, sel, zero_division=0)
    print(f"{name:<18}  {rule:<22}  {tp:>3}  {fp:>3}  {fn:>3}  "
          f"{p:>5.2f}  {r:>5.2f}  {f1:>5.2f}")


print(f"{'method':<18}  {'rule':<22}  {'TP':>3}  {'FP':>3}  {'FN':>3}  "
      f"{'P':>5}  {'R':>5}  {'F1':>5}")
print("-" * 78)

for r in runs:
    s =  r["samples"]
    is_sig = is_signal
    report(r["name"], support_via_ci(s), "95% CI excludes 0", is_sig)

print()
for r in runs:
    if not r["sticky"]: continue
    s = r["samples"]
    is_sig = is_signal
    report(r["name"], support_via_pzero(s), "P(=0) <= 0.5", is_sig)

## 6. Correctness check vs the analytical posterior

For Gaussian prior + Gaussian likelihood + no intercept, the posterior
is exactly multivariate Gaussian. The plain (non-sticky) Boomerang and
Zig-Zag must match the analytical mean and std to within MC error —
this is independent of any HMC comparison.

Healthy values: `max |Δμ| < 0.05`, `mean |Δμ| < 0.01` at the script's
default `n_resample`. The Sticky variants are *expected* to disagree
with the analytic Gaussian — they target a different posterior (with a
spike at zero).

In [ ]:
analytic = next((r for r in runs if r["name"] == "Analytic"), None)
if analytic is None:
    print("No Analytic posterior in this run (likely has_intercept=True).")
else:
    mu_an = analytic["samples"].mean(0)
    sd_an = analytic["samples"].std(0)
    print(f"{'sampler':<14}  {'max |Δμ|':>10}  {'max |Δσ|':>10}  "
          f"{'mean |Δμ|':>10}  {'mean |Δσ|':>10}")
    print("-" * 65)
    for r in runs:
        if r["name"] == "Analytic" or r["sticky"]:
            continue
        s = r["samples"]
        d_mu = np.abs(s.mean(0) - mu_an)
        d_sd = np.abs(s.std(0)  - sd_an)
        print(f"{r['name']:<14}  {d_mu.max():>10.4f}  {d_sd.max():>10.4f}  "
              f"{d_mu.mean():>10.4f}  {d_sd.mean():>10.4f}")